## Partie de Paul

In [7]:
import requests
import pandas as pd
import time

# 1. Liste des années que tu veux récupérer
annees_a_recuperer = ["2020-2021", "2021-2022", "2022-2023", "2023-2024", "2024-2025"]

all_results = []
dataset_id = "fr-en-calendrier-scolaire"
base_url = f"https://data.education.gouv.fr/api/explore/v2.1/catalog/datasets/{dataset_id}/records"

print("🚀 Récupération multi-années via 'refine'...")

for annee in annees_a_recuperer:
    offset = 0
    limit = 100
    print(f"📅 Traitement de l'année : {annee}")
    
    while True:
        params = {
            "limit": limit,
            "offset": offset,
            "refine": f"annee_scolaire:{annee}" # On cible l'année de la boucle
        }
        
        response = requests.get(base_url, params=params)
        if response.status_code != 200:
            print(f"❌ Erreur sur {annee} : {response.status_code}")
            break
            
        data = response.json()
        batch = data.get('results', [])
        
        if not batch:
            break
               
        all_results.extend(batch)
        
        # 'total_annee' nous dit combien il y a de lignes au total pour CETTE année-là
        total_annee = data.get('total_count', 0)
        
        # On calcule combien on en a déjà dans notre liste pour cette année précise
        recu_pour_annee = offset + len(batch)
        
        print(f"   📥 Page chargée : {recu_pour_annee} / {total_annee}")
        
        offset += limit
        time.sleep(0.2) # On peut aller un peu plus vite avec refine
    
# Envoi vers RDS (le reste de ton code ne change pas)
df_final = pd.json_normalize(all_results)
print(f"✅ Terminé ! {len(df_final)} lignes récupérées au total.")

🚀 Récupération multi-années via 'refine'...
📅 Traitement de l'année : 2020-2021
   📥 Page chargée : 100 / 228
   📥 Page chargée : 200 / 228
   📥 Page chargée : 228 / 228
📅 Traitement de l'année : 2021-2022
   📥 Page chargée : 100 / 232
   📥 Page chargée : 200 / 232
   📥 Page chargée : 232 / 232
📅 Traitement de l'année : 2022-2023
   📥 Page chargée : 100 / 229
   📥 Page chargée : 200 / 229
   📥 Page chargée : 229 / 229
📅 Traitement de l'année : 2023-2024
   📥 Page chargée : 100 / 232
   📥 Page chargée : 200 / 232
   📥 Page chargée : 232 / 232
📅 Traitement de l'année : 2024-2025
   📥 Page chargée : 100 / 235
   📥 Page chargée : 200 / 235
   📥 Page chargée : 235 / 235
✅ Terminé ! 1156 lignes récupérées au total.


In [8]:
df_final

,description,population,start_date,end_date,location,zones,annee_scolaire
0,Vacances de Noël,-,2020-12-18T23:00:00+00:00,2021-01-03T23:00:00+00:00,Clermont-Ferrand,Zone A,2020-2021
1,Vacances de Noël,-,2020-12-18T23:00:00+00:00,2021-01-03T23:00:00+00:00,Poitiers,Zone A,2020-2021
2,Vacances d'Hiver,-,2021-02-05T23:00:00+00:00,2021-02-21T23:00:00+00:00,Bordeaux,Zone A,2020-2021
3,Vacances d'Hiver,-,2021-02-05T23:00:00+00:00,2021-02-21T23:00:00+00:00,Clermont-Ferrand,Zone A,2020-2021
4,Vacances d'Hiver,-,2021-02-05T23:00:00+00:00,2021-02-21T23:00:00+00:00,Grenoble,Zone A,2020-2021
...,...,...,...,...,...,...,...
1151,Vacances d'Été,Enseignants,2025-07-04T22:00:00+00:00,2025-08-28T22:00:00+00:00,Créteil,Zone C,2024-2025
1152,Vacances d'Été,Enseignants,2025-07-04T22:00:00+00:00,2025-08-28T22:00:00+00:00,Paris,Zone C,2024-2025
1153,Vacances de Noël,-,2024-12-20T23:00:00+00:00,2025-01-05T23:00:00+00:00,Corse,Corse,2024-2025
1154,Vacances d'Hiver,-,2025-02-14T23:00:00+00:00,2025-03-02T23:00:00+00:00,Corse,Corse,2024-2025


In [9]:
import pandas as pd

# On part de ton df_final (issu de l'API)
# 1. S'assurer que les colonnes sont bien au format date
df_final['start_date'] = pd.to_datetime(df_final['start_date'])
df_final['end_date'] = pd.to_datetime(df_final['end_date'])

# 2. Créer une fonction qui génère la liste des dates entre start et end
def generate_dates(row):
    return pd.date_range(start=row['start_date'], end=row['end_date']).tolist()

# 3. Appliquer la fonction pour créer une nouvelle colonne "liste_dates"
df_final['date'] = df_final.apply(generate_dates, axis=1)

# 4. L'étape magique : EXPLODE
# Cette fonction transforme chaque élément de la liste en une nouvelle ligne
df_unfolded = df_final.explode('date')

# 5. On peut maintenant supprimer les anciennes colonnes start/end si on veut
df_unfolded = df_unfolded.drop(columns=['start_date', 'end_date'])

print(f"✅ Le calendrier est déplié. Nouveau nombre de lignes : {len(df_unfolded)}")

✅ Le calendrier est déplié. Nouveau nombre de lignes : 30612


In [10]:
df_unfolded

,description,population,location,zones,annee_scolaire,date
0,Vacances de Noël,-,Clermont-Ferrand,Zone A,2020-2021,2020-12-18 23:00:00+00:00
0,Vacances de Noël,-,Clermont-Ferrand,Zone A,2020-2021,2020-12-19 23:00:00+00:00
0,Vacances de Noël,-,Clermont-Ferrand,Zone A,2020-2021,2020-12-20 23:00:00+00:00
0,Vacances de Noël,-,Clermont-Ferrand,Zone A,2020-2021,2020-12-21 23:00:00+00:00
0,Vacances de Noël,-,Clermont-Ferrand,Zone A,2020-2021,2020-12-22 23:00:00+00:00
...,...,...,...,...,...,...
1155,Vacances d'Hiver,-,Saint Pierre et Miquelon,Saint Pierre et Miquelon,2024-2025,2025-03-05 23:00:00+00:00
1155,Vacances d'Hiver,-,Saint Pierre et Miquelon,Saint Pierre et Miquelon,2024-2025,2025-03-06 23:00:00+00:00
1155,Vacances d'Hiver,-,Saint Pierre et Miquelon,Saint Pierre et Miquelon,2024-2025,2025-03-07 23:00:00+00:00
1155,Vacances d'Hiver,-,Saint Pierre et Miquelon,Saint Pierre et Miquelon,2024-2025,2025-03-08 23:00:00+00:00


In [11]:
# 2. Le Dico Monstrueux (Mapping Académies/Villes -> Départements)
mapping_dept = {
    'Bordeaux': 'Gironde',
    'Lyon': 'Rhône',
    'Paris': 'Paris',
    'Saint Pierre et Miquelon': 'Saint-Pierre-et-Miquelon',
    'Versailles': 'Yvelines',
    'Créteil': 'Val-de-Marne',
    'Aix-Marseille': 'Bouches-du-Rhône',
    'Nice': 'Alpes-Maritimes',
    'Nantes': 'Loire-Atlantique',
    'Rennes': 'Ille-et-Vilaine',
    'Lille': 'Nord',
    'Montpellier': 'Hérault',
    'Toulouse': 'Haute-Garonne',
    'Strasbourg': 'Bas-Rhin',
    'Nancy-Metz': 'Moselle',
    'Reims': 'Marne',
    'Amiens': 'Somme',
    'Rouen': 'Seine-Maritime',
    'Caen': 'Calvados',
    'Orléans-Tours': 'Loiret',
    'Poitiers': 'Vienne',
    'Limoges': 'Haute-Vienne',
    'Clermont-Ferrand': 'Puy-de-Dôme',
    'Dijon': "Côte-d'Or",
    'Besançon': 'Doubs',
    'Grenoble': 'Isère',
    'Corse': 'Corse-du-Sud',
    'Guadeloupe': 'Guadeloupe',
    'Martinique': 'Martinique',
    'Guyane': 'Guyane',
    'La Réunion': 'La Réunion',
    'Mayotte': 'Mayotte',
    'Wallis et Futuna': 'Wallis-et-Futuna',
    'Polynésie Française': 'Polynésie-Française',
    'Nouvelle Calédonie': 'Nouvelle-Calédonie',
    'Normandie': "Normandie",
    'polynésie': "Polynésie française",
    'Réunion': "Île de la Réunion"
    
}

# 3. Création de la colonne "Departement"
df_unfolded ['Departement'] = df_unfolded ['location'].map(mapping_dept).fillna('Inconnu')

# 4. Nettoyage de la date (Format AAAA/MM/DD sans les heures)
# On convertit en datetime puis on reformate proprement
df_unfolded ['date'] = pd.to_datetime(df_unfolded ['date'], errors='coerce').dt.strftime('%Y/%m/%d')

print("Mission accomplie. Le CSV est carré.")

Mission accomplie. Le CSV est carré.


In [12]:
df_unfolded

,description,population,location,zones,annee_scolaire,date,Departement
0,Vacances de Noël,-,Clermont-Ferrand,Zone A,2020-2021,2020/12/18,Puy-de-Dôme
0,Vacances de Noël,-,Clermont-Ferrand,Zone A,2020-2021,2020/12/19,Puy-de-Dôme
0,Vacances de Noël,-,Clermont-Ferrand,Zone A,2020-2021,2020/12/20,Puy-de-Dôme
0,Vacances de Noël,-,Clermont-Ferrand,Zone A,2020-2021,2020/12/21,Puy-de-Dôme
0,Vacances de Noël,-,Clermont-Ferrand,Zone A,2020-2021,2020/12/22,Puy-de-Dôme
...,...,...,...,...,...,...,...
1155,Vacances d'Hiver,-,Saint Pierre et Miquelon,Saint Pierre et Miquelon,2024-2025,2025/03/05,Saint-Pierre-et-Miquelon
1155,Vacances d'Hiver,-,Saint Pierre et Miquelon,Saint Pierre et Miquelon,2024-2025,2025/03/06,Saint-Pierre-et-Miquelon
1155,Vacances d'Hiver,-,Saint Pierre et Miquelon,Saint Pierre et Miquelon,2024-2025,2025/03/07,Saint-Pierre-et-Miquelon
1155,Vacances d'Hiver,-,Saint Pierre et Miquelon,Saint Pierre et Miquelon,2024-2025,2025/03/08,Saint-Pierre-et-Miquelon


In [ ]:
df_unfolded.columns= df_unfolded.columns.str.lower()

In [ ]:
# df_unfolded.to_csv("Test connection API PaulV2", index = False)

In [17]:
df_unfolded.shape

(30612, 7)